# 📊 Kiểm tra và Thống kê Tập Dữ liệu (Dataset Statistics Check)

Notebook này thực hiện thống kê chi tiết cho hai tập dữ liệu **RDD-2022** và **BharatPotHole** bao gồm:
1. Tỷ lệ hộp bao nhãn ổ gà (pothole labels) trên tổng số nhãn trong tập dữ liệu.
2. Tỷ lệ số lượng ảnh **không chứa nhãn pothole nào**.
3. Kích thước ảnh chủ yếu (predominant resolution) được sử dụng.


In [1]:
import os
from pathlib import Path
from collections import Counter
from PIL import Image
from tqdm.notebook import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
# Tự động xác định PROJECT_ROOT
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

RDD_DIR = RAW_DATA_DIR / 'rdd2022' / 'RDD_SPLIT'
BHARAT_DIR = RAW_DATA_DIR / 'bharatpothole' / 'BharatPotHole' / 'BharatPotHole'

# Cấu hình đồ họa
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')


In [2]:
def find_images(directory):
    directory = Path(directory)
    images = []
    for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
        for img_dir in directory.rglob('images'):
            if img_dir.is_dir():
                images.extend(img_dir.glob(f'*{ext}'))
    return sorted(set(images))

def find_labels(directory):
    directory = Path(directory)
    labels = []
    for labels_dir in directory.rglob('labels'):
        if labels_dir.is_dir():
            labels.extend(labels_dir.glob('*.txt'))
    return sorted(set(labels))


In [3]:
print("=== BẮT ĐẦU PHÂN TÍCH RDD-2022 ===")
rdd_images = find_images(RDD_DIR)
rdd_labels = find_labels(RDD_DIR)

rdd_total_images = len(rdd_images)
rdd_pothole_boxes = 0
rdd_total_boxes = 0
rdd_images_with_pothole = 0

for lbl_path in tqdm(rdd_labels, desc="Đang quét nhãn RDD-2022"):
    has_pothole = False
    box_count = 0
    with open(lbl_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                box_count += 1
                if class_id == 4:  # ID 4 là Pothole trong RDD-2022
                    rdd_pothole_boxes += 1
                    has_pothole = True
    
    rdd_total_boxes += box_count
    if has_pothole:
        rdd_images_with_pothole += 1

# Phân tích kích thước (lấy mẫu 2000 ảnh ngẫu nhiên để tăng tốc độ chạy)
rdd_sizes = Counter()
sample_size = min(2000, len(rdd_images))
import random
random.seed(42)
sampled_rdd_imgs = random.sample(rdd_images, sample_size)

for img_path in tqdm(sampled_rdd_imgs, desc="Đang phân tích kích thước RDD-2022"):
    try:
        with Image.open(img_path) as img:
            rdd_sizes[img.size] += 1
    except:
        pass

rdd_pothole_pct_boxes = (rdd_pothole_boxes / rdd_total_boxes * 100) if rdd_total_boxes > 0 else 0
rdd_pothole_pct_images = (rdd_images_with_pothole / rdd_total_images * 100) if rdd_total_images > 0 else 0
rdd_no_pothole_pct_images = 100 - rdd_pothole_pct_images

print(f"📊 Tổng số ảnh: {rdd_total_images:,}")
print(f"📊 Tổng số hộp nhãn (BBox): {rdd_total_boxes:,}")
print(f"🔥 Nhãn Pothole chiếm: {rdd_pothole_boxes:,} ({rdd_pothole_pct_boxes:.2f}% tổng số BBox)")
print(f"🖼️  Ảnh có chứa ít nhất 1 Pothole: {rdd_images_with_pothole:,} ({rdd_pothole_pct_images:.2f}%)")
print(f"🛡️  Ảnh KHÔNG CHỨA Pothole nào: {rdd_total_images - rdd_images_with_pothole:,} ({rdd_no_pothole_pct_images:.2f}%)")
print(f"📐 3 kích thước ảnh phổ biến nhất: {rdd_sizes.most_common(3)}")


=== BẮT ĐẦU PHÂN TÍCH RDD-2022 ===


Đang quét nhãn RDD-2022:   0%|          | 0/38385 [00:00<?, ?it/s]

Đang phân tích kích thước RDD-2022:   0%|          | 0/2000 [00:00<?, ?it/s]

📊 Tổng số ảnh: 38,385
📊 Tổng số hộp nhãn (BBox): 65,712
🔥 Nhãn Pothole chiếm: 6,544 (9.96% tổng số BBox)
🖼️  Ảnh có chứa ít nhất 1 Pothole: 3,674 (9.57%)
🛡️  Ảnh KHÔNG CHỨA Pothole nào: 34,711 (90.43%)
📐 3 kích thước ảnh phổ biến nhất: [((600, 600), 693), ((720, 720), 355), ((640, 640), 265)]


In [4]:
print("=== BẮT ĐẦU PHÂN TÍCH BHARATPOTHOLE ===")
bharat_images = find_images(BHARAT_DIR)
bharat_labels = find_labels(BHARAT_DIR)

bharat_total_images = len(bharat_images)
bharat_pothole_boxes = 0
bharat_total_boxes = 0
bharat_images_with_pothole = 0

for lbl_path in tqdm(bharat_labels, desc="Đang quét nhãn BharatPotHole"):
    has_pothole = False
    box_count = 0
    with open(lbl_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                box_count += 1
                if class_id == 0:  # ID 0 là Pothole trong BharatPotHole
                    bharat_pothole_boxes += 1
                    has_pothole = True
                    
    bharat_total_boxes += box_count
    if has_pothole:
        bharat_images_with_pothole += 1

# Phân tích kích thước (lấy mẫu 2000 ảnh ngẫu nhiên)
bharat_sizes = Counter()
sample_size_b = min(2000, len(bharat_images))
sampled_bharat_imgs = random.sample(bharat_images, sample_size_b)

for img_path in tqdm(sampled_bharat_imgs, desc="Đang phân tích kích thước BharatPotHole"):
    try:
        with Image.open(img_path) as img:
            bharat_sizes[img.size] += 1
    except:
        pass

bharat_pothole_pct_boxes = (bharat_pothole_boxes / bharat_total_boxes * 100) if bharat_total_boxes > 0 else 0
bharat_pothole_pct_images = (bharat_images_with_pothole / bharat_total_images * 100) if bharat_total_images > 0 else 0
bharat_no_pothole_pct_images = 100 - bharat_pothole_pct_images

print(f"📊 Tổng số ảnh: {bharat_total_images:,}")
print(f"📊 Tổng số hộp nhãn (BBox): {bharat_total_boxes:,}")
print(f"🔥 Nhãn Pothole chiếm: {bharat_pothole_boxes:,} ({bharat_pothole_pct_boxes:.2f}% tổng số BBox)")
print(f"🖼️  Ảnh có chứa ít nhất 1 Pothole: {bharat_images_with_pothole:,} ({bharat_pothole_pct_images:.2f}%)")
print(f"🛡️  Ảnh KHÔNG CHỨA Pothole nào: {bharat_total_images - bharat_images_with_pothole:,} ({bharat_no_pothole_pct_images:.2f}%)")
print(f"📐 3 kích thước ảnh phổ biến nhất: {bharat_sizes.most_common(3)}")


=== BẮT ĐẦU PHÂN TÍCH BHARATPOTHOLE ===


Đang quét nhãn BharatPotHole:   0%|          | 0/7074 [00:00<?, ?it/s]

Đang phân tích kích thước BharatPotHole:   0%|          | 0/2000 [00:00<?, ?it/s]

📊 Tổng số ảnh: 7,074
📊 Tổng số hộp nhãn (BBox): 12,221
🔥 Nhãn Pothole chiếm: 12,221 (100.00% tổng số BBox)
🖼️  Ảnh có chứa ít nhất 1 Pothole: 4,196 (59.32%)
🛡️  Ảnh KHÔNG CHỨA Pothole nào: 2,878 (40.68%)
📐 3 kích thước ảnh phổ biến nhất: [((640, 640), 2000)]


In [5]:
# Lọc các file nhãn có tên chứa "China_Drone" từ danh sách rdd_labels đã quét trước đó
china_drone_labels = [p for p in rdd_labels if "China_Drone" in p.name]
china_drone_potholes = 0
china_drone_total_boxes = 0
china_drone_images_with_pothole = 0

for lbl_path in china_drone_labels:
    has_pothole = False
    with open(lbl_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                china_drone_total_boxes += 1
                if class_id == 4:  # ID 4 là Pothole trong RDD-2022
                    china_drone_potholes += 1
                    has_pothole = True
    if has_pothole:
        china_drone_images_with_pothole += 1

print(f"🇨🇳 Số file nhãn China_Drone: {len(china_drone_labels):,}")
print(f"📊 Tổng số hộp nhãn (BBox) trong China_Drone: {china_drone_total_boxes:,}")
print(f"🔥 Nhãn Pothole (class 4) trong China_Drone: {china_drone_potholes:,} ({china_drone_potholes / china_drone_total_boxes * 100:.2f}% tổng số BBox)")
print(f"🖼️ Số ảnh China_Drone chứa ít nhất 1 Pothole: {china_drone_images_with_pothole:,} ({china_drone_images_with_pothole / len(china_drone_labels) * 100:.2f}%)")


🇨🇳 Số file nhãn China_Drone: 2,401
📊 Tổng số hộp nhãn (BBox) trong China_Drone: 3,840
🔥 Nhãn Pothole (class 4) trong China_Drone: 86 (2.24% tổng số BBox)
🖼️ Số ảnh China_Drone chứa ít nhất 1 Pothole: 64 (2.67%)


In [6]:
from collections import defaultdict
import pandas as pd
import IPython.display as display

# Hàm xác định quốc gia/nguồn từ tên file
def get_country_name(filename):
    if filename.startswith("United_States"):
        return "United States"
    elif filename.startswith("China_Drone"):
        return "China (Drone)"
    elif filename.startswith("China_MotorBike"):
        return "China (MotorBike)"
    # Các quốc gia khác lấy phần tên trước dấu gạch dưới đầu tiên (Ví dụ: Japan, India, Czech, Norway,...)
    return filename.split('_')[0]

# Khởi tạo bảng lưu trữ thống kê
country_stats = defaultdict(lambda: {
    "total_images": 0,
    "images_with_pothole": 0,
    "total_pothole_boxes": 0
})

# Duyệt qua danh sách nhãn RDD-2022 đã quét
for lbl_path in rdd_labels:
    country = get_country_name(lbl_path.name)
    country_stats[country]["total_images"] += 1

    has_pothole = False
    pothole_count = 0

    with open(lbl_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]
        for line in lines:
            parts = line.split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                if class_id == 4:  # ID 4 là Pothole trong RDD-2022
                    pothole_count += 1
                    has_pothole = True

    if has_pothole:
        country_stats[country]["images_with_pothole"] += 1
    country_stats[country]["total_pothole_boxes"] += pothole_count

# Chuyển đổi sang DataFrame để hiển thị bảng số liệu trực quan
df_stats = pd.DataFrame.from_dict(country_stats, orient='index')
df_stats['Tỷ lệ % ảnh có ổ gà'] = (df_stats['images_with_pothole'] / df_stats['total_images'] * 100).round(2)

# Đổi tên cột cho rõ ràng
df_stats.columns = [
    'Tổng số ảnh nhãn',
    'Số ảnh chứa Ổ gà (Lớp 4)',
    'Tổng số nhãn Ổ gà (BBox)',
    'Tỷ lệ % ảnh có ổ gà'
]

# Sắp xếp theo tổng số ảnh giảm dần
df_stats = df_stats.sort_values(by='Tổng số ảnh nhãn', ascending=False)

# Hiển thị bảng kết quả
display.display(df_stats)


,Tổng số ảnh nhãn,Số ảnh chứa Ổ gà (Lớp 4),Tổng số nhãn Ổ gà (BBox),Tỷ lệ % ảnh có ổ gà
Japan,10506,1390,2243,13.23
Norway,8161,256,461,3.14
India,7706,1530,3187,19.85
United States,4805,116,135,2.41
Czech,2829,154,197,5.44
China (Drone),2401,64,86,2.67
China (MotorBike),1977,164,235,8.30


# 📋 Bảng Tổng Kết Kết Quả Phân Tích

| Tiêu chí | Tập dữ liệu RDD-2022 | Tập dữ liệu BharatPotHole |
| :--- | :---: | :---: |
| **Tổng số lượng ảnh** | 38,385 | 7,074 |
| **Tổng số Bounding Box (BBox)** | 65,712 | 12,221 |
| **Số BBox Pothole** | 6,544 | 12,221 |
| **Tỷ lệ % BBox Pothole / Tổng BBox** | **9.96%** | **100%** |
| **Số ảnh có chứa ổ gà (Pothole)** | 3,674 | 4,196 |
| **Tỷ lệ % ảnh có chứa ổ gà** | **9.57%** | **59.32%** |
| **Tỷ lệ % ảnh KHÔNG CHỨA ổ gà** | **90.43%** | **40.68%** |
| **Kích thước ảnh chủ yếu** | 600×600 (~33%), 720×720 (~19%) | 640×640 (100%) |

---

### 💡 Gợi ý xử lý khi gộp dữ liệu để Train model nhận dạng ổ gà:
1. **Lọc nhãn:** Chỉ giữ lại nhãn ổ gà (Class ID = 4) của tập RDD-2022 và chuyển nó về **Class ID = 0** để trùng khớp với nhãn ổ gà của tập BharatPotHole.
2. **Xử lý mất cân bằng nhãn (Imbalance):**
   - RDD-2022 có tới **90.43%** ảnh không có ổ gà (chỉ có vết nứt hoặc không có gì).
   - Nếu bạn đưa toàn bộ 34.711 ảnh không có ổ gà này vào train, mô hình sẽ bị "loãng" và có xu hướng đoán bừa là không có ổ gà.
   - **Đề xuất:** Chỉ lấy khoảng **5% - 10%** lượng ảnh không có ổ gà từ RDD-2022 làm **background images (ảnh nền)** giúp giảm lỗi nhận dạng nhầm vết nứt thành ổ gà. Lượng ảnh nền này tương đương khoảng 2.000 - 3.500 ảnh.
